# 260420_re_lat_ios_attribution_benchmark

| Field | Value |
|-------|-------|
| **Author** | Haewon Yum |
| **Last update** | 2026-04-20 |
| **Objective** | Within iOS RE LAT campaigns active in the last 7 days (any advertiser), quantify what proportion of attributed postbacks is deterministic (deeplink-based) vs probabilistic — benchmark figure to address Netmarble's concern that RE LAT = unreliable attribution. |
| **Scope** | iOS RE LAT campaigns; click-through only (`attribution.reengagement = TRUE`, `attribution.viewthrough = FALSE`); last 7 days; all advertisers |
| **Out of scope** | Netmarble-specific campaigns (none exist for iOS RE LAT); IDFA campaigns; Android RE; UA postbacks; VT postbacks |
| **Key tables** | `moloco-dsp-data-view.postback.pb`, `moloco-ae-view.athena.fact_dsp_core` |
| **Additional context** | Netmarble enabled PA on 2026-04-02; objects to probabilistic attribution as "data integrity harm" (데이터 정합성). Deterministic RE LAT attribution confirmed via deeplink (DDPTICKET-355 filter: `attribution.method = 'DEEPLINK'` OR Adjust `REFERRER`). |

In [23]:
import os
from google.cloud import bigquery
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

In [31]:
client = bigquery.Client(project="moloco-ods")

def run_query(query, label=''):
    df = client.query(query).result().to_dataframe()
    print(f'\u2705 {label}: {len(df)} rows')
    return df

CHART_DIR = os.getcwd()  # charts saved alongside notebook

In [25]:
from datetime import date, timedelta

ANALYSIS_DATE = str(date.today())
START_DATE    = str(date.today() - timedelta(days=7))

print(f'Analysis period: {START_DATE} → {ANALYSIS_DATE}')

Analysis period: 2026-04-13 → 2026-04-20


## Section 1 — RE LAT iOS Campaign Inventory (Last 7 Days)

Identify all RE LAT iOS campaigns with spend in the last 7 days, across all advertisers. Output: campaign count, advertiser count, total spend, total reattributions. Campaign IDs from this section scope the postback pull in Section 2.

In [8]:
# Section 1: RE LAT iOS campaign inventory
# campaign.type = 'APP_REENGAGEMENT', campaign.is_lat = TRUE, campaign.os = 'IOS'
# Partition filter on date_utc is required

q_inventory = f"""
SELECT
  campaign_id,
  campaign.title                    AS campaign_name,
  advertiser_id,
  advertiser.title                  AS advertiser_name,
  advertiser.mmp_bundle_id          AS bundle_id,
  product.app_name,
  product.mmp_name,
  campaign.goal,
  SUM(gross_spend_usd)              AS total_spend_usd,
  SUM(conversions)                  AS total_conversions
FROM `moloco-ae-view.athena.fact_dsp_core`
WHERE date_utc BETWEEN '{START_DATE}' AND '{ANALYSIS_DATE}'
  AND campaign.os     = 'IOS'
  AND campaign.is_lat = TRUE
  AND campaign.type   = 'APP_REENGAGEMENT'
GROUP BY 1, 2, 3, 4, 5, 6, 7, 8
HAVING SUM(gross_spend_usd) > 0
ORDER BY total_spend_usd DESC
"""
df_1 = run_query(q_inventory, label='RE LAT iOS Campaign Inventory')
print(f'Campaigns: {len(df_1)} | Advertisers: {df_1["advertiser_id"].nunique()}')
df_1

✅ RE LAT iOS Campaign Inventory: 25 rows
Campaigns: 25 | Advertisers: 19


,campaign_id,campaign_name,advertiser_id,advertiser_name,bundle_id,app_name,mmp_name,goal,total_spend_usd,total_conversions
0,EhwFqMyIo5zOrRXy,CM_Moloco_iOS_Retargeting_Probabilistic_Test,TkOs2MvWvYALMJso,Moon Active,id406889139,Coin Master,APPSFLYER,OPTIMIZE_REATTRIBUTION_FOR_APP,35315.448026909,19747
1,HIvF2A5z9KMafhxo,RM-MO-WW-iOS-Remarketing-D15plus-High,BZVKTzzvKk1EtzFO,Dream Games,id1482155847,Royal Match,APPSFLYER,OPTIMIZE_REATTRIBUTION_FOR_APP,8848.525138758,71460
2,F7BjGNr4vLpgOaTC,Prog_MO_SBK_US_ALL_NBA_IOS_CV_Test_BetPlaced,TMYION61zkPUh0HF,Sports,com.draftkings.sportsbook,DraftKings Sportsbook & Casino,SINGULAR,OPTIMIZE_CPA_FOR_APP_RE,7616.470253547,6105
3,iNQKZpCTZstdsH6m,luckydefense_ios_moloco_kr_rt_install_lat_260408,t52aeGmi7ov3wppl,Lucky Defense,id6482291732,운빨존많겜,APPSFLYER,OPTIMIZE_REATTRIBUTION_FOR_APP,7023.982060793,432249
4,ltJXzOX6drJrfwPW,MOLOCO_UA_IOS_RET,mFiPldQEiZn1v3rs,Kalshi,id1632713844,Kalshi: Trade News & Sports,APPSFLYER,OPTIMIZE_REATTRIBUTION_FOR_APP,4213.593289272,10660
5,RU226OA4NjIZnAHJ,Raid_RE_Moloco_iOS_RE_US-T1_AppOpen_Deps30_IDF...,DtOQzAMcYG259xkB,Plarium,id1371565796,RAID: Shadow Legends,APPSFLYER,OPTIMIZE_REATTRIBUTION_FOR_APP,3111.064430169,1322
6,J3LSvFGq3ggSBdNF,Moloco-iOS_LAT_RE_ROAS_afpurchase,xgPRFuobLb61FZXH,Jack in the Box,id1283768751,Jack in the Box® Order App,APPSFLYER,OPTIMIZE_REATTRIBUTION_FOR_APP,2832.832588138,218018
7,kyPQCspyiwNBXFGO,RM-MO-WW-iOS-Remarketing-D15plus-Mid,BZVKTzzvKk1EtzFO,Dream Games,id1482155847,Royal Match,APPSFLYER,OPTIMIZE_REATTRIBUTION_FOR_APP,2344.987406724,73782
8,KROcLS4v6eAb0x5d,Echo_MOLOCO_App_Viewcontent_iOS_AppReengage_On...,kwe8WeP2Nh8WgQzv,(ECHO) LINE MANGA,id597088068,LINEマンガ,APPSFLYER,OPTIMIZE_REATTRIBUTION_FOR_APP,2104.717751639,14281
9,k6iLT0k0Khz3Esti,介護_iOS_RE_資格_LAT,PIquDB3ZjIS6NNcL,カイテク,1551769751,介護・看護・保育単発バイトアプリ「カイテク」スキマ求人探し,ADJUST,OPTIMIZE_CPA_FOR_APP_RE,1553.689632963,17044


In [9]:
# Build campaign ID filter for postback queries (Sections 2 & 3)
CAMPAIGN_IDS_SQL = "('" + "', '".join(df_1['campaign_id'].tolist()) + "')"
print(f'{len(df_1)} campaigns: {CAMPAIGN_IDS_SQL[:300]}')

25 campaigns: ('EhwFqMyIo5zOrRXy', 'HIvF2A5z9KMafhxo', 'F7BjGNr4vLpgOaTC', 'iNQKZpCTZstdsH6m', 'ltJXzOX6drJrfwPW', 'RU226OA4NjIZnAHJ', 'J3LSvFGq3ggSBdNF', 'kyPQCspyiwNBXFGO', 'KROcLS4v6eAb0x5d', 'k6iLT0k0Khz3Esti', 'vLzCRqUGzRSIb7Ur', 'KpfHPWZJmkoyz5B4', 'V9gAMwaMeFVBehO3', 'WMigerT4JnHX0gYy', 'cTicyo2cW52zb5t6',


## Section 2 — Attribution Method Distribution (All RE LAT iOS Postbacks)

Count and share (%) of RE LAT click-through postbacks by `attribution.method`. Roll up into: Deterministic (`DEEPLINK` + Adjust `REFERRER`) / Probabilistic (`PROBABILISTIC` + `FINGERPRINT`) / Unknown (`NULL`). Headline figure for Netmarble pitch.

In [34]:
# Section 2: Attribution method distribution — iOS RE LAT click-through postbacks (LAT traffic only)
# Excludes identifier (IDFA) postbacks — proportions computed within true LAT traffic only
# Schema-confirmed fields (live data verification):
#   attribution.view_through  (underscore — NOT viewthrough)
#   mmp.name                  (STRUCT — NOT flat mmp)
#   moloco.campaign_id        (campaign ID field)
#   attribution.method        (lowercase values: deeplink, probabilistic, fingerprint, identifier)

q_method = f"""
WITH base AS (
  SELECT
    attribution.method,
    CASE
      WHEN LOWER(attribution.method) = 'deeplink'
           OR (LOWER(mmp.name) = 'adjust' AND LOWER(attribution.method) = 'referrer')
        THEN 'Deterministic (Deeplink)'
      WHEN LOWER(attribution.method) IN ('probabilistic', 'fingerprint')
        THEN 'Probabilistic'
      ELSE 'Unknown'
    END AS classification
  FROM `moloco-dsp-data-view.postback.pb`
  WHERE DATE(timestamp) BETWEEN '{START_DATE}' AND '{ANALYSIS_DATE}'
    AND attribution.reengagement = TRUE
    AND attribution.view_through = FALSE
    AND attribution.attributed   = TRUE
    AND device.os = 'IOS'
    AND LOWER(attribution.method) != 'identifier'
    AND moloco.campaign_id IN {CAMPAIGN_IDS_SQL}
),
agg AS (
  SELECT method, classification, COUNT(*) AS cnt
  FROM base
  GROUP BY 1, 2
),
total AS (SELECT SUM(cnt) AS grand_total FROM agg)

SELECT
  agg.method,
  agg.classification,
  agg.cnt                                                         AS count,
  ROUND(SAFE_DIVIDE(agg.cnt, total.grand_total) * 100, 2)        AS share_pct
FROM agg CROSS JOIN total
ORDER BY agg.cnt DESC
"""
df_2 = run_query(q_method, label='Attribution Method Distribution (LAT only)')
df_2

✅ Attribution Method Distribution (LAT only): 2 rows


,method,classification,count,share_pct
0,deeplink,Deterministic (Deeplink),703175,72.29
1,probabilistic,Probabilistic,269558,27.71


In [35]:
# Visualize: attribution method distribution (bar chart, share % labels)
color_map = {
    'Deterministic (Deeplink)': '#00CC96',
    'Probabilistic':            '#EF553B',
    'Unknown':                  '#CCCCCC',
}

df_2_plot = df_2.sort_values('count', ascending=False)

fig = px.bar(
    df_2_plot,
    x='classification', y='count',
    color='classification',
    color_discrete_map=color_map,
    text=df_2_plot['share_pct'].apply(lambda x: f'{x:.1f}%'),
    title='iOS RE LAT Attribution Method Distribution — LAT Traffic Only (Last 7 Days)',
    labels={'count': 'Postback Count', 'classification': 'Attribution Method'},
)
fig.update_traces(textposition='outside')
fig.update_layout(showlegend=False, height=450)
fig.show()
fig.write_image(os.path.join(CHART_DIR, 're_lat_ios_attribution_method_distribution.png'), scale=2)

In [36]:
# Section 2b: Bundle-level attribution method breakdown — LAT traffic only (identifier excluded)
# Direct timestamp comparison (no DATE() wrapping) for partition pruning
q_method_bundle = f"""
WITH base AS (
  SELECT
    app.bundle,
    CASE
      WHEN LOWER(attribution.method) = 'deeplink'
           OR (LOWER(mmp.name) = 'adjust' AND LOWER(attribution.method) = 'referrer')
        THEN 'Deterministic (Deeplink)'
      WHEN LOWER(attribution.method) IN ('probabilistic', 'fingerprint')
        THEN 'Probabilistic'
      ELSE 'Unknown'
    END AS classification
  FROM `moloco-dsp-data-view.postback.pb`
  WHERE timestamp >= TIMESTAMP('{START_DATE}')
    AND timestamp <  TIMESTAMP(DATE_ADD(DATE '{ANALYSIS_DATE}', INTERVAL 1 DAY))
    AND attribution.reengagement = TRUE
    AND attribution.view_through = FALSE
    AND attribution.attributed   = TRUE
    AND device.os = 'IOS'
    AND LOWER(attribution.method) != 'identifier'
    AND moloco.campaign_id IN {CAMPAIGN_IDS_SQL}
),
bundle_agg AS (
  SELECT bundle, classification, COUNT(*) AS cnt
  FROM base
  GROUP BY 1, 2
),
bundle_total AS (
  SELECT bundle, SUM(cnt) AS bundle_total FROM bundle_agg GROUP BY bundle
)

SELECT
  ba.bundle,
  ba.classification,
  ba.cnt                                                          AS count,
  ROUND(SAFE_DIVIDE(ba.cnt, bt.bundle_total) * 100, 2)           AS share_pct,
  bt.bundle_total                                                 AS total_postbacks
FROM bundle_agg ba
JOIN bundle_total bt USING (bundle)
ORDER BY bt.bundle_total DESC, ba.classification
"""
df_2b = run_query(q_method_bundle, label='Bundle-level Attribution Method (LAT only)')

# Join app_name from Section 1
bundle_map = (
    df_1[['bundle_id', 'app_name', 'advertiser_name']]
    .drop_duplicates(subset='bundle_id')
    .rename(columns={'bundle_id': 'bundle'})
)
df_2b = df_2b.merge(bundle_map, on='bundle', how='left')
df_2b['label'] = df_2b['app_name'].where(df_2b['app_name'].notna(), df_2b['bundle'])

# Flag any com.* bundles (Android-format) appearing in iOS data for awareness
android_format = df_2b[df_2b['bundle'].str.startswith('com.', na=False)]['bundle'].unique()
if len(android_format) > 0:
    print(f"\u26a0\ufe0f  com.* bundles in iOS postbacks: {android_format.tolist()}")
    print("   These are iOS apps registered with Android-format bundle IDs in their MMP.")

# Summary: deterministic share per app — all apps shown (0% for fully probabilistic ones)
det = (
    df_2b.pivot_table(
        index=['bundle', 'label', 'total_postbacks'],
        columns='classification',
        values='share_pct',
        aggfunc='sum',
    )
    .reset_index()
    .rename_axis(None, axis=1)
)
det['det_share_pct'] = det.get('Deterministic (Deeplink)', 0).fillna(0)
det = (
    det[['label', 'bundle', 'det_share_pct', 'total_postbacks']]
    .sort_values('det_share_pct', ascending=False)
    .reset_index(drop=True)
)

print(f"\nDeterministic share range: {det['det_share_pct'].min():.1f}% \u2013 {det['det_share_pct'].max():.1f}%")
print(f"Median: {det['det_share_pct'].median():.1f}% across {len(det)} apps")
det

✅ Bundle-level Attribution Method (LAT only): 16 rows

Deterministic share range: 0.0% – 100.0%
Median: 31.7% across 10 apps


,label,bundle,det_share_pct,total_postbacks
0,Kalshi: Trade News & Sports,id1632713844,100.00,18633
1,Jack in the Box® Order App,id1283768751,99.40,290519
2,화해 (대한민국 1등 뷰티 앱),id940056100,99.19,2856
3,Royal Match,id1482155847,96.19,195751
4,운빨존많겜,id6482291732,47.41,431489
5,TOEIC®対策ならSantaアルク AI学習でスコアUP,id1148006701,16.07,280
6,Coin Master,id406889139,0.35,11860
7,GoldenHoYeah Slots-Slots Games,id1020109860,0.00,5402
8,LINEマンガ,id597088068,0.00,15854
9,Caesars Palace Online Casino,id6446176419,0.00,89


In [40]:
# Section 2b follow-up: bundles with near-zero deterministic share (~100% probabilistic)
# Threshold: det_share_pct < 5% — adjust if needed
PROB_THRESHOLD = 5.0

prob_bundles = det[det['det_share_pct'] < PROB_THRESHOLD]['bundle'].tolist()
print(f"Bundles with <{PROB_THRESHOLD}% deterministic share: {len(prob_bundles)}")

# Full attribution breakdown for these bundles from df_2b
prob_detail = (
    df_2b[df_2b['bundle'].isin(prob_bundles)]
    [['label', 'bundle', 'classification', 'count', 'share_pct', 'total_postbacks']]
    .sort_values(['total_postbacks', 'classification'], ascending=[False, True])
    .reset_index(drop=True)
)
prob_detail

Bundles with <5.0% deterministic share: 4


,label,bundle,classification,count,share_pct,total_postbacks
0,LINEマンガ,id597088068,Probabilistic,15854,100.00,15854
1,Coin Master,id406889139,Deterministic (Deeplink),42,0.35,11860
2,Coin Master,id406889139,Probabilistic,11818,99.65,11860
3,GoldenHoYeah Slots-Slots Games,id1020109860,Probabilistic,5402,100.00,5402
4,Caesars Palace Online Casino,id6446176419,Probabilistic,89,100.00,89


In [41]:
# Stacked bar: attribution method share per app, ordered by deterministic share (highest → lowest)
app_order = (
    det.sort_values('det_share_pct', ascending=False)['label'].tolist()
)

fig_bundle = px.bar(
    df_2b,
    x='label', y='share_pct',
    color='classification',
    color_discrete_map=color_map,
    barmode='stack',
    category_orders={'label': app_order},
    text=df_2b['share_pct'].apply(lambda x: f'{x:.0f}%' if x >= 8 else ''),
    title='iOS RE LAT Attribution Method by App — LAT Traffic Only, Ordered by Deterministic Share (Last 7 Days)',
    labels={'share_pct': 'Share (%)', 'label': 'App', 'classification': 'Attribution Method'},
    hover_data={'bundle': True},
)
fig_bundle.update_layout(height=500, yaxis_ticksuffix='%', xaxis_tickangle=-30)
fig_bundle.show()
fig_bundle.write_image(os.path.join(CHART_DIR, 're_lat_ios_attribution_by_bundle.png'), scale=2)

## Section 3 — Daily Trend: Deterministic vs Probabilistic Share (Past 7 Days)

Group RE LAT postbacks by date × classification (Deterministic / Probabilistic / Unknown). Confirm the deterministic share is consistent day-over-day. Flag single-day drops that may indicate deeplink breakage.

In [38]:
# Section 3: Daily trend — Deterministic vs Probabilistic (LAT traffic only, identifier excluded)
q_trend = f"""
WITH base AS (
  SELECT
    DATE(timestamp) AS date,
    CASE
      WHEN LOWER(attribution.method) = 'deeplink'
           OR (LOWER(mmp.name) = 'adjust' AND LOWER(attribution.method) = 'referrer')
        THEN 'Deterministic (Deeplink)'
      WHEN LOWER(attribution.method) IN ('probabilistic', 'fingerprint')
        THEN 'Probabilistic'
      ELSE 'Unknown'
    END AS classification
  FROM `moloco-dsp-data-view.postback.pb`
  WHERE DATE(timestamp) BETWEEN '{START_DATE}' AND '{ANALYSIS_DATE}'
    AND attribution.reengagement = TRUE
    AND attribution.view_through = FALSE
    AND attribution.attributed   = TRUE
    AND device.os = 'IOS'
    AND LOWER(attribution.method) != 'identifier'
    AND moloco.campaign_id IN {CAMPAIGN_IDS_SQL}
),
daily_agg AS (
  SELECT date, classification, COUNT(*) AS cnt
  FROM base
  GROUP BY 1, 2
),
daily_total AS (
  SELECT date, SUM(cnt) AS day_total FROM daily_agg GROUP BY date
)

SELECT
  da.date,
  da.classification,
  da.cnt                                                    AS count,
  ROUND(SAFE_DIVIDE(da.cnt, dt.day_total) * 100, 2)        AS share_pct
FROM daily_agg da
JOIN daily_total dt USING (date)
ORDER BY da.date, da.classification
"""
df_3 = run_query(q_trend, label='Daily Attribution Method Trend (LAT only)')
df_3

✅ Daily Attribution Method Trend (LAT only): 16 rows


,date,classification,count,share_pct
0,2026-04-13,Deterministic (Deeplink),82211,78.25
1,2026-04-13,Probabilistic,22856,21.75
2,2026-04-14,Deterministic (Deeplink),83577,76.65
3,2026-04-14,Probabilistic,25459,23.35
4,2026-04-15,Deterministic (Deeplink),84707,76.44
5,2026-04-15,Probabilistic,26110,23.56
6,2026-04-16,Deterministic (Deeplink),82515,74.84
7,2026-04-16,Probabilistic,27742,25.16
8,2026-04-17,Deterministic (Deeplink),87688,71.19
9,2026-04-17,Probabilistic,35482,28.81


In [39]:
# Stacked bar: daily deterministic vs probabilistic share (%)
fig = px.bar(
    df_3,
    x='date', y='share_pct',
    color='classification',
    color_discrete_map=color_map,
    barmode='stack',
    text=df_3['share_pct'].apply(lambda x: f'{x:.0f}%' if x >= 5 else ''),
    title='iOS RE LAT Attribution Method — Daily Share (Last 7 Days)',
    labels={'share_pct': 'Share (%)', 'date': 'Date', 'classification': 'Attribution Method'},
)
fig.update_layout(height=450, yaxis_ticksuffix='%')
fig.show()
fig.write_image(os.path.join(CHART_DIR, 're_lat_ios_attribution_daily_trend.png'), scale=2)

## Section 4 — KOR Office RE LAT iOS Campaign Inventory (Last 6 Months)

Identify all RE LAT iOS campaigns run by KOR office advertisers in the last 6 months (`advertiser.office = 'KOR'`). Campaign IDs scope Sections 5–7.

In [42]:
START_6M = str(date.today() - timedelta(days=182))  # ~6 months

q_kor_inventory = f"""
SELECT
  campaign_id,
  campaign.title                    AS campaign_name,
  advertiser_id,
  advertiser.title                  AS advertiser_name,
  advertiser.mmp_bundle_id          AS bundle_id,
  product.app_name,
  product.mmp_name,
  campaign.goal,
  MIN(date_utc)                     AS first_seen,
  MAX(date_utc)                     AS last_seen,
  SUM(gross_spend_usd)              AS total_spend_usd,
  SUM(conversions)                  AS total_conversions
FROM `moloco-ae-view.athena.fact_dsp_core`
WHERE date_utc BETWEEN '{START_6M}' AND '{ANALYSIS_DATE}'
  AND campaign.os     = 'IOS'
  AND campaign.is_lat = TRUE
  AND campaign.type   = 'APP_REENGAGEMENT'
  AND advertiser.office = 'KOR'
GROUP BY 1, 2, 3, 4, 5, 6, 7, 8
HAVING SUM(gross_spend_usd) > 0
ORDER BY total_spend_usd DESC
"""
df_4 = run_query(q_kor_inventory, label='KOR Office RE LAT iOS Campaign Inventory (6M)')
print(f'Campaigns: {len(df_4)} | Advertisers: {df_4["advertiser_id"].nunique()}')
df_4

✅ KOR Office RE LAT iOS Campaign Inventory (6M): 10 rows
Campaigns: 10 | Advertisers: 9


,campaign_id,campaign_name,advertiser_id,advertiser_name,bundle_id,app_name,mmp_name,goal,first_seen,last_seen,total_spend_usd,total_conversions
0,JCbqtZCd2NehCzLL,KR_RE_moloco_maugrowth_AppOpen_IOS_251003,Voql38wJkmDNzXbW,당근마켓,id1018769995,당근,APPSFLYER,OPTIMIZE_REATTRIBUTION_FOR_APP,2025-10-21,2026-04-20,128116.299595050,10802544
1,OPx66EKUtSgGX9Sz,E_JP_moloco_rt_ios_manual_LAT_fp_260223_all,jp41i8qyihbZe4X5,(ECHO) 로켓나우,com.cpone.customer,Rocket Now : 出前/フードデリバリー,BRANCH,OPTIMIZE_REATTRIBUTION_FOR_APP,2026-02-23,2026-04-20,45671.725960100,811798
2,iNQKZpCTZstdsH6m,luckydefense_ios_moloco_kr_rt_install_lat_260408,t52aeGmi7ov3wppl,Lucky Defense,id6482291732,운빨존많겜,APPSFLYER,OPTIMIZE_REATTRIBUTION_FOR_APP,2026-04-08,2026-04-20,12065.209911693,617110
3,V9gAMwaMeFVBehO3,MLC_KR_iOS_RE_AppOpen_260305,kRmtgHyEC830uwpH,Riiid,id1148006701,TOEIC®対策ならSantaアルク AI学習でスコアUP,APPSFLYER,OPTIMIZE_REATTRIBUTION_FOR_APP,2026-03-05,2026-04-20,6612.134215097,4404
4,hta46uizLDS0hsx7,E_Moloco_DA_LAT_UV_JP_iOS_2603_mtnd_N,VYtkS7p5CHEpOwtZ,MUSINSA JAPAN,id1637547116,MUSINSA – 韓国人気ブランドが集まるファッション通販,APPSFLYER,OPTIMIZE_REATTRIBUTION_FOR_APP,2026-03-09,2026-04-20,3256.628405728,46894
5,cTicyo2cW52zb5t6,RE-appopen-ios-intrabound-lat,Os7oojpjTo8JwHIt,야놀자,id436731843,NOL(야놀자),APPSFLYER,OPTIMIZE_REATTRIBUTION_FOR_APP,2026-03-12,2026-04-20,3151.789120831,529726
6,QuITYoLWQkEBnCXS,luckydefense_ios_moloco_kr_rt_cpa,t52aeGmi7ov3wppl,Lucky Defense,id6482291732,운빨존많겜,APPSFLYER,OPTIMIZE_CPA_FOR_APP_RE,2025-10-30,2026-02-01,1570.615184400,73598
7,F0BfNf9zgEHWLsc0,SSG_Moloco_AO,YWBVDlZDyeQLCOIL,SSG.COM_IM,com.emart.ssg,SSG.COM - 쓱닷컴,AIRBRIDGE,OPTIMIZE_REATTRIBUTION_FOR_APP,2026-02-23,2026-03-01,195.516787364,2641
8,eUbvGIh25iypjWzW,kr_ios_re_251112_moloco_ace_open_sus,yC0uTzHsfZwVpayQ,DRIMAGE - Architect,6698862298,아키텍트: 랜드 오브 엑자일,ADJUST,OPTIMIZE_REATTRIBUTION_FOR_APP,2025-11-12,2025-12-08,169.335819046,2
9,VtatZr2NGfWL65eu,moloco_기존(유입)_iOS,LoRCqRutktxzhF61,Hwahae,id940056100,화해 (대한민국 1등 뷰티 앱),APPSFLYER,OPTIMIZE_REATTRIBUTION_FOR_APP,2026-04-14,2026-04-20,2.134457171,4


In [43]:
# Build campaign ID filter for KOR office postback queries (Sections 5-7)
KOR_CAMPAIGN_IDS_SQL = "('" + "', '".join(df_4['campaign_id'].tolist()) + "')"
print(f'{len(df_4)} KOR campaigns scoped for Sections 5-7')

10 KOR campaigns scoped for Sections 5-7


## Section 5 — Attribution Method Distribution (KOR Office, 6 Months)

Same analysis as Section 2 but scoped to KOR office RE LAT iOS campaigns over 6 months. IDFA excluded — proportions within LAT traffic only.

In [46]:
print(f"KOR campaign IDs: {len(df_4)}")
print(KOR_CAMPAIGN_IDS_SQL[:200])

KOR campaign IDs: 10
('JCbqtZCd2NehCzLL', 'OPx66EKUtSgGX9Sz', 'iNQKZpCTZstdsH6m', 'V9gAMwaMeFVBehO3', 'hta46uizLDS0hsx7', 'cTicyo2cW52zb5t6', 'QuITYoLWQkEBnCXS', 'F0BfNf9zgEHWLsc0', 'eUbvGIh25iypjWzW', 'VtatZr2NGfWL65eu')


In [47]:
q_kor_method = f"""
WITH base AS (
  SELECT
    attribution.method,
    CASE
      WHEN LOWER(attribution.method) = 'deeplink'
           OR (LOWER(mmp.name) = 'adjust' AND LOWER(attribution.method) = 'referrer')
        THEN 'Deterministic (Deeplink)'
      WHEN LOWER(attribution.method) IN ('probabilistic', 'fingerprint')
        THEN 'Probabilistic'
      ELSE 'Unknown'
    END AS classification
  FROM `moloco-dsp-data-view.postback.pb`
  WHERE timestamp >= TIMESTAMP('{START_6M}')
    AND timestamp <  TIMESTAMP(DATE_ADD(DATE '{ANALYSIS_DATE}', INTERVAL 1 DAY))
    AND attribution.reengagement = TRUE
    AND attribution.view_through = FALSE
    AND attribution.attributed   = TRUE
    AND device.os = 'IOS'
    AND LOWER(attribution.method) != 'identifier'
    AND moloco.campaign_id IN {KOR_CAMPAIGN_IDS_SQL}
),
agg AS (
  SELECT method, classification, COUNT(*) AS cnt
  FROM base
  GROUP BY 1, 2
),
total AS (SELECT SUM(cnt) AS grand_total FROM agg)

SELECT
  agg.method,
  agg.classification,
  agg.cnt                                                         AS count,
  ROUND(SAFE_DIVIDE(agg.cnt, total.grand_total) * 100, 2)        AS share_pct
FROM agg CROSS JOIN total
ORDER BY agg.cnt DESC
"""
df_5 = run_query(q_kor_method, label='KOR Office Attribution Method Distribution (6M)')
df_5

✅ KOR Office Attribution Method Distribution (6M): 2 rows


,method,classification,count,share_pct
0,deeplink,Deterministic (Deeplink),12947149,94.51
1,probabilistic,Probabilistic,752447,5.49


In [48]:
fig = px.bar(
    df_5.sort_values('count', ascending=False),
    x='classification', y='count',
    color='classification',
    color_discrete_map=color_map,
    text=df_5.sort_values('count', ascending=False)['share_pct'].apply(lambda x: f'{x:.1f}%'),
    title='iOS RE LAT Attribution Method — KOR Office, LAT Traffic Only (Last 6 Months)',
    labels={'count': 'Postback Count', 'classification': 'Attribution Method'},
)
fig.update_traces(textposition='outside')
fig.update_layout(showlegend=False, height=450)
fig.show()
fig.write_image(os.path.join(CHART_DIR, 're_lat_ios_kor_attribution_method.png'), scale=2)

## Section 6 — Bundle-level Breakdown (KOR Office, 6 Months)

Deterministic share per app bundle for KOR office advertisers. Ordered by deterministic share descending. Flags bundles with ~0% deterministic (deeplink not configured).

In [49]:
q_kor_bundle = f"""
WITH base AS (
  SELECT
    app.bundle,
    CASE
      WHEN LOWER(attribution.method) = 'deeplink'
           OR (LOWER(mmp.name) = 'adjust' AND LOWER(attribution.method) = 'referrer')
        THEN 'Deterministic (Deeplink)'
      WHEN LOWER(attribution.method) IN ('probabilistic', 'fingerprint')
        THEN 'Probabilistic'
      ELSE 'Unknown'
    END AS classification
  FROM `moloco-dsp-data-view.postback.pb`
  WHERE timestamp >= TIMESTAMP('{START_6M}')
    AND timestamp <  TIMESTAMP(DATE_ADD(DATE '{ANALYSIS_DATE}', INTERVAL 1 DAY))
    AND attribution.reengagement = TRUE
    AND attribution.view_through = FALSE
    AND attribution.attributed   = TRUE
    AND device.os = 'IOS'
    AND LOWER(attribution.method) != 'identifier'
    AND moloco.campaign_id IN {KOR_CAMPAIGN_IDS_SQL}
),
bundle_agg AS (
  SELECT bundle, classification, COUNT(*) AS cnt
  FROM base GROUP BY 1, 2
),
bundle_total AS (
  SELECT bundle, SUM(cnt) AS bundle_total FROM bundle_agg GROUP BY bundle
)

SELECT
  ba.bundle,
  ba.classification,
  ba.cnt                                                          AS count,
  ROUND(SAFE_DIVIDE(ba.cnt, bt.bundle_total) * 100, 2)           AS share_pct,
  bt.bundle_total                                                 AS total_postbacks
FROM bundle_agg ba
JOIN bundle_total bt USING (bundle)
ORDER BY bt.bundle_total DESC, ba.classification
"""
df_6 = run_query(q_kor_bundle, label='KOR Office Bundle-level Attribution (6M)')

# Join app_name from Section 4
bundle_map_kor = (
    df_4[['bundle_id', 'app_name', 'advertiser_name']]
    .drop_duplicates(subset='bundle_id')
    .rename(columns={'bundle_id': 'bundle'})
)
df_6 = df_6.merge(bundle_map_kor, on='bundle', how='left')
df_6['label'] = df_6['app_name'].where(df_6['app_name'].notna(), df_6['bundle'])

# Deterministic share summary — all apps (0% for fully probabilistic)
det_kor = (
    df_6.pivot_table(
        index=['bundle', 'label', 'total_postbacks'],
        columns='classification', values='share_pct', aggfunc='sum',
    )
    .reset_index().rename_axis(None, axis=1)
)
det_kor['det_share_pct'] = det_kor.get('Deterministic (Deeplink)', 0).fillna(0)
det_kor = (
    det_kor[['label', 'bundle', 'det_share_pct', 'total_postbacks']]
    .sort_values('det_share_pct', ascending=False).reset_index(drop=True)
)
print(f"Deterministic share range: {det_kor['det_share_pct'].min():.1f}% – {det_kor['det_share_pct'].max():.1f}%")
print(f"Median: {det_kor['det_share_pct'].median():.1f}% across {len(det_kor)} apps")
det_kor

✅ KOR Office Bundle-level Attribution (6M): 10 rows
Deterministic share range: 25.0% – 99.5%
Median: 74.6% across 5 apps


,label,bundle,det_share_pct,total_postbacks
0,화해 (대한민국 1등 뷰티 앱),id940056100,99.50,78264
1,당근,id1018769995,96.51,13026238
2,MUSINSA – 韓国人気ブランドが集まるファッション通販,id1637547116,74.63,51527
3,운빨존많겜,id6482291732,47.69,541150
4,TOEIC®対策ならSantaアルク AI学習でスコアUP,id1148006701,24.99,2417


In [50]:
app_order_kor = det_kor.sort_values('det_share_pct', ascending=False)['label'].tolist()

fig_kor_bundle = px.bar(
    df_6,
    x='label', y='share_pct',
    color='classification',
    color_discrete_map=color_map,
    barmode='stack',
    category_orders={'label': app_order_kor},
    text=df_6['share_pct'].apply(lambda x: f'{x:.0f}%' if x >= 8 else ''),
    title='iOS RE LAT Attribution Method by App — KOR Office, LAT Traffic Only (Last 6 Months)',
    labels={'share_pct': 'Share (%)', 'label': 'App', 'classification': 'Attribution Method'},
    hover_data={'bundle': True},
)
fig_kor_bundle.update_layout(height=500, yaxis_ticksuffix='%', xaxis_tickangle=-30)
fig_kor_bundle.show()
fig_kor_bundle.write_image(os.path.join(CHART_DIR, 're_lat_ios_kor_attribution_by_bundle.png'), scale=2)

## Section 7 — Monthly Trend: Deterministic vs Probabilistic Share (KOR Office, 6 Months)

Group KOR office RE LAT postbacks by month × classification. Confirms deterministic share is stable or improving over the 6-month window.

In [51]:
q_kor_trend = f"""
WITH base AS (
  SELECT
    FORMAT_DATE('%Y-%m', DATE(timestamp)) AS month,
    CASE
      WHEN LOWER(attribution.method) = 'deeplink'
           OR (LOWER(mmp.name) = 'adjust' AND LOWER(attribution.method) = 'referrer')
        THEN 'Deterministic (Deeplink)'
      WHEN LOWER(attribution.method) IN ('probabilistic', 'fingerprint')
        THEN 'Probabilistic'
      ELSE 'Unknown'
    END AS classification
  FROM `moloco-dsp-data-view.postback.pb`
  WHERE timestamp >= TIMESTAMP('{START_6M}')
    AND timestamp <  TIMESTAMP(DATE_ADD(DATE '{ANALYSIS_DATE}', INTERVAL 1 DAY))
    AND attribution.reengagement = TRUE
    AND attribution.view_through = FALSE
    AND attribution.attributed   = TRUE
    AND device.os = 'IOS'
    AND LOWER(attribution.method) != 'identifier'
    AND moloco.campaign_id IN {KOR_CAMPAIGN_IDS_SQL}
),
monthly_agg AS (
  SELECT month, classification, COUNT(*) AS cnt
  FROM base GROUP BY 1, 2
),
monthly_total AS (
  SELECT month, SUM(cnt) AS month_total FROM monthly_agg GROUP BY month
)

SELECT
  ma.month,
  ma.classification,
  ma.cnt                                                     AS count,
  ROUND(SAFE_DIVIDE(ma.cnt, mt.month_total) * 100, 2)       AS share_pct
FROM monthly_agg ma
JOIN monthly_total mt USING (month)
ORDER BY ma.month, ma.classification
"""
df_7 = run_query(q_kor_trend, label='KOR Office Monthly Attribution Trend (6M)')
df_7

✅ KOR Office Monthly Attribution Trend (6M): 14 rows


,month,classification,count,share_pct
0,2025-10,Deterministic (Deeplink),146763,96.66
1,2025-10,Probabilistic,5074,3.34
2,2025-11,Deterministic (Deeplink),654762,96.90
3,2025-11,Probabilistic,20933,3.10
4,2025-12,Deterministic (Deeplink),1483723,96.39
5,2025-12,Probabilistic,55627,3.61
6,2026-01,Deterministic (Deeplink),2490525,96.46
7,2026-01,Probabilistic,91525,3.54
8,2026-02,Deterministic (Deeplink),2727103,96.17
9,2026-02,Probabilistic,108463,3.83


In [52]:
fig_kor_trend = px.bar(
    df_7,
    x='month', y='share_pct',
    color='classification',
    color_discrete_map=color_map,
    barmode='stack',
    text=df_7['share_pct'].apply(lambda x: f'{x:.0f}%' if x >= 8 else ''),
    title='iOS RE LAT Attribution Method — KOR Office Monthly Trend (Last 6 Months)',
    labels={'share_pct': 'Share (%)', 'month': 'Month', 'classification': 'Attribution Method'},
)
fig_kor_trend.update_layout(height=450, yaxis_ticksuffix='%')
fig_kor_trend.show()
fig_kor_trend.write_image(os.path.join(CHART_DIR, 're_lat_ios_kor_attribution_monthly_trend.png'), scale=2)